# Day12. 사전 학습 모형 (Pre-Training Model)

> NLP · 김종찬 강사님 · 2026-10-01 · BERT / GPT / 의료 멀티모달(CLIP · BLIP)

### 🎯 학습 목표
1. **사전 학습(Pre-Training)** 과 **미세 조정(Fine-Tuning)** 의 차이를 설명할 수 있다.
2. BERT의 입력 구조(3가지 임베딩)와 사전 학습 과제 **MLM · NSP** 를 이해하고 실행한다.
3. 문장 임베딩 + 코사인 유사도로 **검색형 챗봇** 을 만든다.
4. GPT로 문장을 생성하고, **디코딩 하이퍼파라미터**(Greedy, Beam, Top-K, Temperature, 반복 억제)의 효과를 비교한다.
5. 의료 영상 데이터(VQA-RAD)에 **CLIP · BLIP** 을 적용해 제로샷 분류, 텍스트→영상 검색, 캡셔닝, VQA를 해 보고, Fine-Tuning 모형을 점검한다.

### 📚 목차
0. NLP 사전 학습 개요 · 1. BERT (MLM · NSP) · 2. BERT 챗봇 · 3. GPT 문장 생성 · 4. GPT 하이퍼파라미터
**📊 도식화 자료 (p71~80)**
5. 의료 멀티모달 데이터 · 6. CLIP 제로샷 · 7. 제로샷의 한계 · 8. CLIP 영상 검색 · 9. BLIP 캡셔닝 · 10. VQA · 11. Fine-Tuning 모형 비교

### ⚙️ 실습 환경
- VS Code + Jupyter, 커널 **Python 3.12.10** (`.venv` 를 찾지 못해 기본 파이썬에 설치해 진행)
- 처음 실행 시 모형 약 8개(합계 수 GB)를 Hugging Face 에서 다운로드

## 0. NLP Pre Training (사전 학습)

- 대규모의 언어 데이터를 이용하여 미리 만들어진 모델을 활용해 다른 모델이나 어플리케이션에 적용하는 기법
- 신경망 알고리즘의 **전이학습 (Transfer Learning)** 을 활용 — 특정 대상을 훈련시킨 모델을 다른 대상에도 적용하여 예측할 수 있도록 바꾸는 딥러닝 기법
- **자기지도학습 (Self-Supervised Learning)** : 정답 라벨 없이 입력 자체에서 학습 신호(가중치 정보)를 만드는 방식
- 자연어 처리 Pre Training
    1. **Transformer Encoder Model (BERT)**
        - Bidirectional Encoder Representations from Transformers
        - 자연어 이해 (Word Embedding / Word2Vec / MLM / Classification ..) 에 많이 활용
        - 문서/문장 분류 / NER 개체명 인식 / 질의응답(추출) / 검색 ...
        - BERT 핵심 학습 방식
            - MLM (Masked Language Model) : 문장 내 일부 단어를 가리고 양방향으로 문맥을 파악하며 단어를 예측
            - NSP (Next Sentence Prediction) : 두 문장이 문맥적으로 서로 이어지는 문장인지 예측
            - **Bidirectional(양방향) 문맥을 동시에 고려하는 것이 GPT 와의 핵심 차이**
    2. **Transformer Decoder Model (GPT)**
        - Generative Pre-trained Transformer
        - 모델이 주어진 문장이나 데이터를 바탕으로 다음 단어를 예측하고 생성
        - 생성형 모델에 주로 사용 — 대화, 요약, 생성, 추론 등의 생성형 작업 전반
        - 단방향(왼쪽 → 오른쪽)으로만 문맥을 읽어 다음 단어를 예측 (**AutoRegressive, 자기회귀** : 앞에서 생성한 단어를 다시 입력으로 넣어 다음 단어를 생성) → 이것이 BERT(양방향)와의 핵심 차이
        - 학습할 때는 병렬로 처리되지만, 추론할 때는 한 단어씩 순차적으로 진행
    3. **Encoder-Decoder (T5 / BART)**
        - 기계 번역 / 요약 / 생성형 질의응답 등
        - T5 : 모든 NLP 태스크를 Text-to-Text 형식으로 통일하여 처리 (예: 번역도 요약도 '글을 넣으면 글이 나온다')
        - BART : 노이즈가 추가된(망가뜨린) 문장을 원래대로 복원하는 방식으로 학습

> 💡 **보충 : '단방향'과 '병렬'은 서로 다른 이야기**
> - **단방향** = 어텐션이 **앞쪽 단어만** 본다는 뜻. 학습·추론 **모두** 단방향이다. 학습 때는 **Masked Self-Attention**(뒤쪽 단어를 가리는 마스크)으로 미래 단어를 못 보게 막는다.
> - **병렬 vs 순차** = 계산 방식의 차이
>   - 학습 : 정답 문장 전체를 한 번에 넣고 모든 위치의 '다음 단어'를 동시에 예측 → **병렬** (Day10 Teacher Forcing 과 같은 원리)
>   - 추론 : 정답이 없으니 한 단어를 만들고 그걸 다시 넣어 다음 단어를 만듦 → **순차**

> 💡 **보충 : 사전 학습 vs 미세 조정**
>
> | 구분 | 사전 학습 (Pre-Training) | 미세 조정 (Fine-Tuning) |
> |---|---|---|
> | 데이터 | 라벨 없는 대규모 텍스트 | 라벨 있는 소규모 과제 데이터 |
> | 목적 | 언어의 일반적 패턴 학습 | 특정 과제에 맞춤 |
> | 비용 | 매우 큼 (기업·연구소) | 작음 (개인도 가능) |
> | 오늘 수업 | `klue/bert-base`, `skt/kogpt2-base-v2`, CLIP, BLIP | `ayyuce/blip-vqa-rad` |

### 💡 Hugging Face 란?

**Hugging Face** = 사전 학습 모형과 데이터셋을 공유하는 플랫폼 (AI 계의 GitHub). 👉 https://huggingface.co

- 코드에서 `'만든곳/모형이름'` 문자열만 넣으면 자동으로 다운로드된다 (처음 1회만, 이후엔 내 컴퓨터 캐시 `C:\Users\사용자\.cache\huggingface` 에서 불러옴)
- 각 모형 페이지(**Model Card**)에서 학습 데이터, 용도, 라이선스, 사용 예시를 확인할 수 있다

| 오늘 쓰는 모형 | 페이지 | 용도 |
|---|---|---|
| `klue/bert-base` | https://huggingface.co/klue/bert-base | 한국어 BERT (MLM · NSP) |
| `sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens` | https://huggingface.co/sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens | 다국어 문장 임베딩 (챗봇) |
| `skt/kogpt2-base-v2` | https://huggingface.co/skt/kogpt2-base-v2 | 한국어 GPT-2 (문장 생성) |
| `openai/clip-vit-base-patch32` | https://huggingface.co/openai/clip-vit-base-patch32 | 일반 CLIP |
| `flaviagiammarino/pubmed-clip-vit-base-patch32` | https://huggingface.co/flaviagiammarino/pubmed-clip-vit-base-patch32 | 의료 CLIP |
| `Salesforce/blip-image-captioning-base` | https://huggingface.co/Salesforce/blip-image-captioning-base | 이미지 캡셔닝 |
| `Salesforce/blip-vqa-base` | https://huggingface.co/Salesforce/blip-vqa-base | 일반 VQA |
| `ayyuce/blip-vqa-rad` | https://huggingface.co/ayyuce/blip-vqa-rad | 의료 VQA 미세 조정 |

**알아두면 좋은 것**
- **LOAD REPORT** : 모형을 불러올 때 출력되는 표. `UNEXPECTED` = 파일에는 있는데 이번 구조에선 안 쓰는 가중치(대부분 무시 가능), `MISSING` = 구조에는 필요한데 파일에 없어 **무작위로 채워진** 가중치 → 결과가 엉망이 될 수 있으니 반드시 확인 (11번에서 실제 사례)
- **symlink 경고** (`cache-system uses symlinks...`) : Windows 에서 저장 방식 관련 경고일 뿐, 실행에는 문제 없음

In [ ]:
# ===== 0. Install Packages =====


# 처음 한 번만 실행 — 설치 후 주석 처리 (수업 때는 Python 3.12.10 에 아래 한 줄로 설치)
# %pip install transformers torch sentence-transformers datasets pillow matplotlib

# transformers : Hugging Face 사전 학습 모형 라이브러리
# torch : 모형을 실제로 계산하는 딥러닝 엔진 (PyTorch)
# sentence-transformers : 문장 → 벡터 변환 전용 라이브러리 (챗봇 파트)
# datasets / pillow / matplotlib : 데이터셋 다운로드 / 이미지 처리 / 시각화 (멀티모달 파트)
# tf-keras : TensorFlow 관련 모듈 호환용 (강사님 코드에 포함, PyTorch 만 쓸 땐 없어도 동작)
# %pip install tf-keras


# → %pip 는 '지금 노트북이 쓰는 파이썬(커널)'에 설치하는 명령이라, 다른 파이썬에 잘못 설치되는 일을 막아주겠다
# → 설치 직후에는 커널을 다시 시작(Restart)해야 새 패키지를 인식하겠다

## 1. BERT

- 위키피디아(25억 개 단어)와 BooksCorpus(8억 개 단어)를 레이블이 없는 텍스트 데이터로 사전 훈련한 모형
- Fine-Tuning 을 이용해 태스크(과제)에 맞는 형태로 사용
- **Fine Tuning** : 이미 설계된 모델이 다른 작업(다른 데이터)에 대해 파라미터를 재조정하기 위한 추가 훈련 과정
- **BERT 구조** — 단어를 Embedding 하기 위한 세 가지 영역이 존재
    1. **Word Embedding** (입력 만들기)
        - **Word Piece Embedding** : 단어를 벡터로 변환 (서브워드 단위로 분리하여 변환)
            - 미등록 단어(OOV) 문제를 줄이는 효과 — 예) `"playing"` → `"play"` + `"##ing"`
            - 예) `"나는 10월이 참 좋아"` → `[나] + [##는] + [10월] + [##이] + [참] + [좋] + [##아]`
        - **Positional Embedding** : 단어의 위치 정보를 계산/반환
        - **Segment Embedding** : 각 문장/문단을 구분하는 정보를 계산
    2. **Bidirectional Transformer** (사전 학습) — 문장의 앞뒤 문맥을 동시에 고려하여 각 단어의 의미를 계산
        - **MLM** : 입력 텍스트의 약 15% 를 무작위로 가리고(Masking) 모델이 해당 자리를 채우며 학습
        - **NSP** : 두 문장이 주어졌을 때 두 번째 문장이 첫 번째 문장의 논리적인 후속 문장인지 판별 (IsNext / NotNext) → 질의응답(QA), 자연어 추론(NLI) 성능 향상에 기여

> 💡 **보충 : 서브워드 (Subword) 란?**
> - 단어를 **더 작은 의미 조각**으로 나눈 단위. 예) `playing` → `play` + `##ing`, `김치찌개` → `김치` + `##찌개`
> - **왜 쓰나?** 단어 단위 사전은 신조어·오타·희귀어를 모두 담을 수 없음 → 사전에 없는 단어(**OOV, Out-Of-Vocabulary**)가 `[UNK]` 로 뭉개짐. 조각 단위면 처음 보는 단어도 아는 조각들의 조합으로 표현 가능
> - **`##` 표시** = "앞 조각에 **붙어서 이어지는** 조각" 이라는 뜻. 단어의 **첫 조각에는 붙지 않고, 이어지는 조각 앞**에 붙는다
>   → 필기 예시의 `[좋다##]` 는 규칙상 `[좋]` + `[##아]` 처럼 쓰는 것이 맞다 (실제 분리는 토크나이저 사전에 따라 다름)
> - **WordPiece** : 자주 함께 나오는 글자 조각을 하나로 합쳐 가며 사전을 만드는 알고리즘 (BERT 가 사용)

> 💡 **보충 : Positional · Segment Embedding 이 필요한 이유**
> - **Positional** : Transformer 는 모든 단어를 **동시에** 처리해서, 그대로 두면 순서를 모른다 (`나는 너를` ≠ `너는 나를` 을 구분 못 함) → 각 자리마다 위치 벡터를 더해 순서를 알려준다
> - **Segment** : 두 문장을 한 줄로 넣을 때 앞 문장은 `E_A`, 뒤 문장은 `E_B` 를 더해 구분 → NSP 처럼 **두 문장의 관계**를 볼 때 꼭 필요
> - 최종 입력 = **토큰 임베딩 + 세그먼트 임베딩 + 위치 임베딩** (세 벡터를 더한 값, 도식화 D03 참고)

> 💡 **보충 : BERT 크기와 특수 토큰**
>
> | 모델 | 층 수 | 은닉 차원 | 어텐션 헤드 | 파라미터 |
> |---|---|---|---|---|
> | BERT-base | 12 | 768 | 12 | 약 1.1억 |
> | BERT-large | 24 | 1024 | 16 | 약 3.4억 |
>
> - `[CLS]` : 문장 맨 앞 토큰. 문장 전체를 대표하는 벡터로 분류·NSP 판단에 사용
> - `[SEP]` : 문장과 문장의 경계 (`[CLS] 문장A [SEP] 문장B [SEP]`)
> - `[MASK]` : MLM 에서 가려진 단어 자리
> - **KLUE** (Korean Language Understanding Evaluation) : 한국어 이해 성능 평가 벤치마크. `klue/bert-base` 는 이 팀이 공개한 **한국어 BERT-base**

### 1-1. MLM (Masked Language Model)

- 텍스트에서 임의로 일부 단어를 가리고, 빈칸을 모델이 채우도록 학습
  - 원본 문장 : 나는 고양이와 강아지 중 어떤 걸 키울지 고민이야.
  - 마스킹 실시 : 나는 `[MASK]` 와 `[MASK]` 중 어떤 걸 키울지 고민이야.
- 사전 훈련을 위해 입력 텍스트의 **15%** 단어를 Random Masking 처리

> 💡 **보충 : 마스킹 규칙** — 전체 토큰의 **15%** 를 골라, 그중 **80% 는 `[MASK]`**, **10% 는 무작위 단어**, **10% 는 그대로** 둔다. 실제 사용(Fine-Tuning) 때는 `[MASK]` 가 나오지 않으므로, 모든 자리를 성실히 보도록 섞어주는 장치다.

In [ ]:
# ===== 1-1. Fill-Mask Pipeline =====


# pipeline : 모형 불러오기 → 입력 전처리 → 예측 → 결과 정리를 한 줄로 묶어주는 간편 함수
from transformers import pipeline

# 'fill-mask' : [MASK] 자리에 들어갈 단어를 예측하는 작업 이름
# model='klue/bert-base' : Hugging Face 에서 한국어 BERT 를 다운로드
fill = pipeline('fill-mask', model='klue/bert-base')


# → MLM 으로 사전 학습된 한국어 BERT 를 빈칸 채우기 도구(fill)로 준비하겠다
# → LOAD REPORT 의 bert.pooler.* UNEXPECTED 는 빈칸 채우기에 안 쓰는 층이라 무시해도 되겠다

In [ ]:
# ===== 1-2. Predict [MASK] =====


# 문장 속 [MASK] 토큰 자리에 올 단어 후보를 확률(score) 높은 순으로 5개 반환
fill("AI 헬스케어 팀 프로젝트는 정말 [MASK] 하다")


# → 결과의 token_str 은 예측 단어, score 는 그 단어일 확률, sequence 는 채워진 완성 문장이다
# → 수업 결과 : 어마어마(18.4%) > 대단(12.6%) > 흥미진진(9.5%) > 중요 > 드라마틱 — 모두 문법·의미상 자연스럽겠다
# → 앞('정말')과 뒤('하다')를 모두 보고 고른 결과라 이것이 '양방향', 파인튜닝 없이 사전 학습만으로 가능하다

### 1-2. NSP (Next Sentence Prediction)

- 두 개의 문장을 준 뒤, 문장이 이어지는지 아닌지 판단하는 방식. 랜덤하게 문장을 섞어서 학습하여 **문장 간의 관계를 이해**하도록 학습하는 방법
- BERT 는 **50 : 50** 비율로 실제 이어지는 문장과 Random 으로 이어 붙인 두 문장을 주고 훈련

> 💡 **보충 : NSP 학습 데이터와 판별 방식** (도식화 D05)
> - 학습 쌍의 **50% 는 실제 다음 문장 (IsNext)**, **50% 는 다른 글에서 무작위로 가져온 문장 (NotNext)**
> - 정답은 '원래 글에서 이어졌는가'로 자동으로 생기므로 사람이 라벨을 달 필요가 없음 → 자기지도학습
> - 판별 흐름 : `[CLS] 문장A [SEP] 문장B [SEP]` → BERT 인코더 → **`[CLS]` 자리의 출력 벡터** → 분류층(출력 2개) → softmax → `0 = 이어짐`, `1 = 안 이어짐`
> - 참고 : 이후 나온 **RoBERTa** 는 NSP 를 빼고도 성능이 유지되어, NSP 효과에 대해서는 논쟁이 있다

In [ ]:
# ===== 1-3. Load NSP Model =====


# BertForNextSentencePrediction : BERT 위에 '이어지는 문장인가?'를 판단하는 출력층이 붙은 모형
from transformers import BertForNextSentencePrediction
# AutoTokenizer : 모형 이름에 맞는 토크나이저(문장 → 토큰 번호 변환기)를 자동으로 골라줌
from transformers import AutoTokenizer
# torch : 텐서 계산, softmax 등에 사용
import torch

# 사용할 사전 학습 모형 이름 = 한국어 BERT-base (위 1번 설명 참고)
model_name = 'klue/bert-base'
# 입력 문장을 토큰화하는 도구
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 앞/뒤 문장의 연결 여부를 비교하는 모형
model = BertForNextSentencePrediction.from_pretrained(model_name)


# → 같은 klue/bert-base 를 이번엔 NSP 출력층이 달린 형태로 불러오겠다
# → LOAD REPORT 에서 cls.predictions.* 가 UNEXPECTED 로 나오는 건 MLM 출력층이라 NSP 에선 안 쓰는 것이고, MISSING 이 있는지만 확인하겠다

In [ ]:
# ===== 1-4. NSP Score Function =====


# 두 문장 a, b 가 이어질 확률을 계산하는 함수
def nsp_score(a, b):
    # tokenizer(a, b) : [CLS] a [SEP] b [SEP] 형태로 두 문장을 한 입력으로 묶음 (세그먼트 A/B 정보 포함)
    # return_tensors='pt' : 결과를 PyTorch 텐서로 반환
    input1 = tokenizer(a, b, return_tensors='pt')
    # torch.no_grad() : 학습이 아닌 예측이므로 기울기(gradient) 계산을 끄고 메모리·속도 절약
    with torch.no_grad():
        # model(**input1) : 딕셔너리를 풀어서(input_ids, token_type_ids, attention_mask) 모형에 전달
        # .logits : softmax 전의 원점수 [이어짐 점수, 안 이어짐 점수]
        result = model(**input1).logits
        # softmax : 원점수를 합이 1인 확률로 변환 / dim=-1 : 마지막 축 기준
        # squeeze() : 크기 1인 차원 제거 → shape (1, 2) 를 (2,) 로
        probs = torch.softmax(result, dim=-1).squeeze()
    # 0번 = 이어지는 문장(IsNext), 1번 = 이어지지 않음(NotNext)
    return "두 문장이 이어질 확률", probs[0], "이어지지 않을 확률", probs[1]


# → 두 문장을 [SEP] 으로 붙여 넣고, 0번(이어짐)·1번(안 이어짐) 확률을 돌려주겠다

In [ ]:
# ===== 1-5. NSP Test =====


# 자연스럽게 이어지는 두 문장
a = '오늘 집에서 김치찌개 먹었어!!!'
b = '근데 커피를 못마셨네 ㅠㅠ'
print(nsp_score(a, b))

# 주제가 전혀 다른 두 문장
a = '난 오늘 그녀에게 고백할꺼야'
b = '오늘 자연어 처리 해야하는데...'
print(nsp_score(a, b))


# → 수업 결과 : 첫 쌍 이어짐 0.6572 / 안 이어짐 0.3428, 둘째 쌍 이어짐 0.0029 / 안 이어짐 0.9971
# → 같은 '먹고 마시기' 이야기는 이어짐 쪽, 화제가 완전히 다른 쌍은 거의 확실히 안 이어짐으로 판단하겠다

#### 📌 NSP 결과 해석 자세히

`('두 문장이 이어질 확률', tensor(0.0029), '이어지지 않을 확률', tensor(0.9971))`

1. **숫자의 의미** : 모델이 "두 번째 문장이 첫 번째 문장의 실제 다음 문장일 확률"을 **0.29%**, "아닐 확률"을 **99.71%** 로 계산했다. 두 값은 softmax 를 거쳐 **항상 합이 1** 이 된다.
2. **왜 이렇게 나왔나** : '고백' 과 '자연어 처리 과제' 는 사전 학습 때 본 수많은 글에서 **연달아 나오는 일이 거의 없는 화제**다. BERT 는 `[CLS]` 벡터에 두 문장의 관계를 요약하는데, 여기서 연결 근거(같은 주제, 이어지는 접속 표현, 대명사 연결 등)를 찾지 못한 것.
3. **첫 번째 쌍이 65.7% 에 그친 이유** : '김치찌개 → 커피' 는 '먹고 마시기' 로 이어지지만 음식 종류가 바뀌고 `근데` 로 화제를 트는 문장이라 **완전히 확신하지는 못한** 것. NSP 확률은 '맞다/틀리다'가 아니라 **얼마나 자연스러운 연결인지의 정도**로 읽는 게 좋다.
4. **`tensor(...)` 표시** : PyTorch 텐서라서 붙는 표시. 숫자만 보고 싶으면 `probs[0].item()` 처럼 `.item()` 을 붙인다.
5. **활용** : 두 문장의 관계를 봐야 하는 **질의응답(QA)**, **자연어 추론(NLI, 두 문장이 모순/함의 관계인지)** 같은 과제의 바탕이 된다.

## 2. BERT 를 이용한 챗봇 (Sentence Embedding + 코사인 유사도)

- Q&A 데이터를 Fine-Tuning 에 활용하여 적용 (강의 자료)
- 어제(Day11) 썼던 Q&A 챗봇 데이터를 가져와, 질문들의 관계를 **사전 학습 모형으로** 파악하는 실습

> 💡 **보충 : 이 실습은 엄밀히 말하면 Fine-Tuning 이 아니다** — 코드에서 모형의 가중치를 다시 학습(`fit`, `train`)하는 부분이 없다. 사전 학습된 모형을 **그대로** 써서 질문들을 벡터로 바꾸고 검색만 한다 (**특징 추출, Feature Extraction** 방식). 진짜 Fine-Tuning 이라면 Q&A 쌍으로 모형을 추가 학습해 가중치를 바꾼다. 강의에서 말한 'Fine-Tuning 에 활용'은 '사전 학습 모형을 내 데이터에 적용한다'는 넓은 의미로 이해하자.

**아이디어** : 답을 새로 '생성'하지 않고, **미리 준비된 Q&A 중 가장 비슷한 질문을 찾아 그 답(A)을 돌려주는 검색형 챗봇**

1. **문장 임베딩 (Sentence Embedding)** : 문장 전체를 의미가 담긴 숫자 벡터 하나로 변환 (비슷한 뜻 → 비슷한 벡터)
2. 데이터의 모든 질문(Q)을 미리 벡터로 바꿔 저장
3. 사용자 질문도 벡터로 바꾼 뒤 **코사인 유사도**로 가장 가까운 질문을 찾음

**코사인 유사도 (Cosine Similarity)** : 두 벡터가 가리키는 **방향이 얼마나 같은지** (1 = 같은 방향, 0 = 무관, -1 = 반대)

$$\cos(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

> 💡 **보충 : sentence-transformers 라이브러리**
> - 일반 BERT 는 **단어(토큰)마다** 벡터를 내놓는다. 문장 하나를 벡터 하나로 만들려면 따로 처리가 필요하다.
> - **sentence-transformers (Sentence-BERT, SBERT)** 는 토큰 벡터들을 평균(mean pooling) 내어 문장 벡터를 만들고, **"뜻이 비슷한 문장은 벡터도 가깝게"** 되도록 추가 학습해 둔 라이브러리다.
> - `model.encode(문장)` 한 줄이면 문장 → 벡터. 검색, 챗봇, 중복 문장 찾기, 군집화에 많이 쓴다. 👉 https://www.sbert.net

In [ ]:
# ===== 2-1. Compatibility Patch =====


# sentence-transformers 일부 버전이 최신 transformers 에서 삭제된 TFPreTrainedModel 을 찾다가 오류가 남
# → 같은 이름의 빈 클래스를 끼워 넣어(더미) import 오류를 막는 임시 처리
import os
# torchvision(이미지용 패키지)을 불러오지 않도록 설정 — 불필요한 의존성 오류 방지
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
import transformers

# 아무 기능 없는 빈 클래스 ( ... 은 '내용 없음'을 뜻하는 파이썬 문법)
class _DummyTFPreTrainedModel: ...

# transformers 안에 사라진 이름을 빈 클래스로 다시 등록
transformers.TFPreTrainedModel = _DummyTFPreTrainedModel


# → 라이브러리 버전 차이로 생기는 ImportError 를 '빈 껍데기 클래스'로 우회하겠다 (모형 계산에는 영향 없음)

In [ ]:
# ===== 2-2. Import & Load Data =====


import pandas as pd
import numpy as np
# SentenceTransformer : 문장을 고정 길이 벡터로 바꿔주는 모형 클래스
from sentence_transformers import SentenceTransformer
# tqdm : 반복문 진행률 막대 표시 / .auto : 노트북·터미널 환경에 맞게 자동 선택
from tqdm.auto import tqdm

# [참고] tqdm.pandas() 의 progress_apply 는 pandas 3.0 에서 호환성 문제 → tqdm 을 직접 사용

# index_col=0 : CSV 의 첫 번째 열을 인덱스로 사용 (저장 시 생긴 번호 열)
# ※ 경로는 이 노트북 위치 기준 — 데이터 폴더 위치에 맞게 수정
df1 = pd.read_csv('data/09_QA_ChatBot_Data/Chat_Data.csv', index_col=0)
print(df1.shape)
df1.head(2)


# → Q(질문)·A(답변) 쌍으로 된 챗봇 데이터를 불러와 크기와 앞부분을 확인하겠다
# → 수업 중 이 셀에서 SyntaxError 가 났던 건 코드 셀에 필기를 '#' 없이 적었기 때문 — 필기는 마크다운 셀이나 # 주석으로 쓰겠다

In [ ]:
# ===== 2-3. Load Sentence Model =====


# 100개 언어로 학습된 다국어 문장 임베딩 모형 (한국어 포함, XLM-RoBERTa 기반)
# nli-stsb : 문장 의미 유사도 데이터(NLI, STS-B)로 추가 학습 / mean-tokens : 토큰 벡터들의 평균을 문장 벡터로 사용
url = 'sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens'
model = SentenceTransformer(url)

# 첫 번째 질문 확인 → '12시 땡!'
print(df1['Q'][0])
# model.encode(문장) : 문장 → 768차원 벡터
model.encode(df1['Q'][0])


# → '문장 의미가 비슷하면 벡터도 가깝게' 학습된 모형을 불러와, '12시 땡!' 이 768개 숫자로 된 벡터 하나로 바뀌는 것을 확인하겠다
# → 사전 학습된 가중치가 있기 때문에 처음 보는 문장도 바로 의미 벡터로 바꿀 수 있겠다

In [ ]:
# ===== 2-4. Encode All Questions =====


# df1['Q'] 의 모든 질문을 하나씩 벡터로 변환해 새 열 embedding 에 저장
# tqdm(..., desc=...) : 진행률 막대에 표시할 이름
df1['embedding'] = [model.encode(x) for x in tqdm(df1['Q'], desc="문장 인코딩")]


# → 답변 검색 때마다 계산하지 않도록 모든 질문 벡터를 미리 만들어 두겠다 (몇 분 걸릴 수 있음)
# → 수업 중 NameError(model / tqdm is not defined)는 위 셀들을 실행하지 않고 건너뛰었을 때 생기므로, 셀은 위에서부터 순서대로 실행하겠다

In [ ]:
# ===== 2-5. Cosine Similarity & Answer Function =====


# 코사인 유사도 : 내적(np.dot) ÷ (A 길이 × B 길이)
# np.linalg.norm : 벡터의 길이(크기)
def cos_sim(A, B):
    return np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B))

# 새 질문 Q 와 가장 비슷한 질문을 찾아 그 답변을 반환
def return_answer(Q):
    # 입력 문장을 벡터로 변환
    embedd = model.encode(Q)
    # 저장된 모든 질문 벡터와 유사도를 계산해 Score 열에 저장
    df1['Score'] = df1['embedding'].apply(lambda x: cos_sim(x, embedd))
    # idxmax() : Score 가 가장 큰 행의 인덱스 → 그 행의 답변(A) 반환
    return df1.loc[df1['Score'].idxmax()]['A']


# → 사용자 질문 벡터와 가장 방향이 같은 질문을 골라, 그 질문에 짝지어진 답변을 돌려주겠다

#### 📌 `return_answer` 조각조각 해석

| 조각 | 의미 |
|---|---|
| `embedd = model.encode(Q)` | 사용자 질문 `'배고프당'` → 768차원 벡터 1개 |
| `df1['embedding']` | 미리 저장해 둔 모든 질문 벡터가 들어 있는 열 |
| `.apply(lambda x: cos_sim(x, embedd))` | 그 열의 **각 행 벡터 x** 마다 `cos_sim(x, embedd)` 를 계산 (`lambda x:` = 이름 없는 한 줄짜리 함수) |
| `df1['Score'] = ...` | 계산된 유사도들을 새 열 `Score` 로 저장 → 행마다 "사용자 질문과 얼마나 비슷한지" 점수가 붙음 |
| `df1['Score'].idxmax()` | Score 가 **가장 큰 행의 인덱스(번호)** 를 반환 (값이 아니라 위치) |
| `df1.loc[인덱스]` | 그 번호의 **행 전체**(Q, A, embedding, Score)를 꺼냄 |
| `[...]['A']` | 그 행에서 **답변(A) 칸만** 꺼내 반환 |

→ 한 문장으로 : **"모든 질문과 점수를 매기고 → 1등 질문의 행을 찾아 → 그 행의 답변을 돌려준다"**

In [ ]:
# ===== 2-6. Chatbot Test =====


print(return_answer('배고프당'))
print(return_answer('이 과정이 끝나면 그녀에게 고백할꺼야!!'))
print(return_answer('마니또가 선물로 차를 선물해줬으면 좋겠어!'))
# 데이터에 비슷한 질문이 없는 경우 (강사님 추가 질문)
print(return_answer('김강사 이번에 만든 시뮬레이터의 이름을 뭘로 붙이면 좋을까?'))
print(return_answer('우리 AI 헬스케어 7기 조교님의 이름은?'))


# → 수업 결과 : '얼른 맛난 음식 드세요.' / '오 미리 축하드려요.' / '비용을 따져보세요.' / '저는 사람으로 태어나고 싶어요.' / '위로봇이요.'
# → 앞의 세 질문은 데이터에 비슷한 질문이 있어 그럴듯한 답이 나오겠다
# → 뒤의 두 질문은 데이터에 없는 내용이라, '이름'이 들어간 엉뚱한 질문의 답을 가져오겠다 — 검색형 챗봇은 데이터에 있는 답만 돌려줄 수 있다는 한계

## 3. GPT (Generative Pre-trained Transformer)

- Transformer 의 **디코더(Decoder)만** 쌓은 모형 → 문장을 **생성**하는 데 강함
- 자기회귀 방식으로 작동 : 이전 단어들을 기반으로 다음 단어를 예측 / 단방향 문장 처리
  - (강의 자료의 'Self Regressive' 는 정식 용어로 **Auto-Regressive** — auto = 자기 자신)
- 사전 학습 과제 : **다음 단어 예측** (앞의 단어들만 보고 다음 단어를 맞힘 = 단방향, Autoregressive)
- `skt/kogpt2-base-v2` : SKT 가 공개한 한국어 GPT-2

| 구분 | BERT | GPT |
|---|---|---|
| 구조 | Transformer 인코더 | Transformer 디코더 |
| 문맥 방향 | 양방향 (앞+뒤) | 단방향 (앞만) |
| 사전 학습 과제 | MLM + NSP | 다음 단어 예측 |
| 잘하는 일 | 이해 (분류·검색·QA) | 생성 (대화·요약·글쓰기) |

In [ ]:
# ===== 3-1. Load KoGPT2 =====


# GPT2LMHeadModel : GPT-2 본체 + 다음 단어 확률을 내는 출력층(LM Head)
from transformers import GPT2LMHeadModel
# PreTrainedTokenizerFast : 빠른(Rust 기반) 토크나이저 — KoGPT2 는 이 클래스로 불러옴
from transformers import PreTrainedTokenizerFast

model_name = 'skt/kogpt2-base-v2'
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)


# → 한국어 문장 생성용 GPT-2 모형과 짝이 되는 토크나이저를 불러오겠다

In [ ]:
# ===== 3-2. Generate Text Function =====


def generate_text(input_text, model, tokenizer):
    # encode : 문장 → 토큰 번호 텐서
    input_data = tokenizer.encode(input_text, return_tensors='pt')
    # generate : 다음 토큰을 하나씩 반복 예측해 이어 붙임
    # max_length=50 : 입력 포함 최대 50 '토큰' / num_return_sequences=1 : 결과 1개
    outputs = model.generate(input_data, max_length=50, num_return_sequences=1)
    # decode : 토큰 번호 → 한글 / skip_special_tokens=True : </s> 같은 특수 토큰 제거
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return output_text


# → 입력 문장 뒤에 이어질 내용을 기본 설정(Greedy Search)으로 생성하겠다

#### 📌 `generate_text` 각 부분의 의미

| 코드 | 의미 |
|---|---|
| `tokenizer.encode(input_text, return_tensors='pt')` | 문장을 토큰 번호 목록으로 바꾸고 PyTorch 텐서(`pt`)로 반환. shape = (문장 1개, 토큰 수) |
| `model.generate(...)` | '다음 토큰 예측 → 뒤에 붙이기 → 다시 예측'을 반복하는 생성 함수 |
| `max_length=50` | 생성을 멈추는 길이. **단어 수가 아니라 토큰 수**이고, **입력 문장의 토큰까지 포함**한 전체 길이다. 입력이 15토큰이면 새로 만드는 건 최대 35토큰. (새로 만들 토큰 수만 정하려면 `max_new_tokens`) |
| `num_return_sequences=1` | **결과 시퀀스(생성 결과물)를 1개** 돌려달라는 뜻. 맞다, 한 개만 생성한다. 단, '시퀀스 1개' 안에 문장이 여러 개 들어 있을 수 있다 (아래 설명) |
| `outputs[0]` | 결과가 여러 개일 수 있어 목록으로 오므로 첫 번째를 꺼냄 |
| `tokenizer.decode(...)` | 토큰 번호 → 사람이 읽는 글자로 복원 |
| `skip_special_tokens=True` | `</s>`(문장 끝), `<pad>`(빈칸 채움) 같은 **모형 내부용 특수 토큰을 결과 글자에서 빼라**는 뜻. False 면 `...먹지?</s>` 처럼 기호가 섞여 나온다 |

In [ ]:
# ===== 3-3. Generation Test =====


input_text = "오늘은 배고픈데 뭐먹지? 아 그러면,"
print(generate_text(input_text, model, tokenizer))

input_text2 = "다음 최종 프로젝트 주제는 뭘로 하는게 좋을까? 내 생각엔"
print(generate_text(input_text2, model, tokenizer))


# → 수업 결과 : '아니야. 배고픈데?' 가 세 번 반복, '그건 내가 결정할 문제지.' 가 계속 반복되겠다
# → 기본 설정(Greedy)의 반복 문제를 다음 파트에서 하이퍼파라미터로 개선하겠다

#### 📌 왜 같은 말을 반복하고, 왜 EOS 를 못 만날까?

- **Greedy Search 의 함정** : 기본값은 매 시점 **확률 1등 단어만** 고른다. '배고픈데?' 를 한 번 쓰고 나면, 앞 문맥에 '배고픈데?' 가 있으니 다음에도 비슷한 흐름이 가장 그럴듯해지고 → 다시 같은 말을 고르고 → 문맥이 더 강화되는 **되먹임 고리**에 빠진다. 이를 **반복 퇴화(Degeneration)** 라고 한다.
- 수업에서 말한 **'자유도'** = 단어 선택에 얼마나 무작위성을 허용하느냐
  - 자유도 낮음(Greedy) : 맥락에는 맞지만 똑같은 말만 반복
  - 자유도 높음(Sampling, 높은 Temperature) : 다양한 단어를 내뱉지만 엉뚱한 소리가 늘어남
- **EOS 를 못 만나는 이유** : 생성은 ① 문장 끝 토큰 **EOS(`</s>`)** 가 나오거나 ② `max_length` 에 닿으면 멈춘다. 반복 고리 안에서는 매번 반복 구절이 EOS 보다 확률이 높아서 EOS 가 1등이 될 기회가 없다 → 결국 **max_length 에서 강제로 잘린다** (결과가 문장 중간에서 끊겨 있는 이유).

## 4. GPT 하이퍼파라미터 튜닝 (디코딩 전략)

GPT 는 매 단계 '다음 단어 확률표'를 만든다. **그 표에서 단어를 어떻게 고를지** 정하는 것이 디코딩 전략이다.

### 4-1. 수업 필기 정리

- **Greedy Search** : 각 시점에서 **가장 높은 확률을 가진 단어 하나**를 탐색/선택하는 기법
  - 빠르게 문장을 생성할 수 있지만 항상 최적의 결과를 보장하지는 않음 → 문장이 반복되거나 다양성이 부족할 수 있음
- **Beam Search** : 현 시점에서 **가능성 있는 후보 여러 개를 유지**한 채 단어를 탐색
  - 매번 1등만 고르는 대신 후보 문장 k개(`num_beams`)를 동시에 키워 가다가, **문장 전체 확률**이 가장 높은 것을 선택
  - 더 나은 결과를 얻을 수 있지만 계산 비용이 증가 (후보 수만큼 계산)
- **Top-K Sampling** : 각 시점에서 확률이 높은 **상위 k개 후보 중에서 무작위로** 선택
  - 문장의 다양성을 높이고 자연스러운 문장을 구성할 때 사용. 대신 문맥에 맞지 않는 단어가 선택될 수 있음
  - 1등만 고르는 대신, 후보군 밖의 단어는 제외하고 후보군 안에서는 **확률에 비례한 추첨**으로 고른다
  - 수업 실험 : `top_k` 를 **30 → 10** 으로 낮추면 후보군이 줄어 다양성이 줄고 질문과 비슷한 말이 나옴 / **100** 으로 늘리면 다양성이 높아져 딴소리가 많아짐
- **Repetition Penalty** : 이미 생성된 단어에 **패널티를 부여**해 반복을 방지
  - 높은 패널티 : 모델이 출력하는 결과의 **다양성**을 보장 → 자연스러운 문장 / 문맥에 적합하지 않은 출력이 나올 수 있음
  - 낮은 패널티 : 모델이 출력하는 결과의 **일관성**을 보장 → 일관된 문장 / 똑같은 말이나 단어를 반복
- **Temperature Sampling** : 모델이 계산한 확률 분포를 더 뾰족하게/평평하게 조정해 샘플링
  - **온도 낮음 (T < 1)** : 확률 차이를 **극대화** → 높은 확률 단어가 더 잘 뽑힘 → 일관성 ↑, 다양성 ↓
  - **온도 높음 (T > 1)** : 확률을 **평탄화** → 확률이 낮던 단어도 뽑힐 가능성 ↑ → 다양성 ↑, 일관성 ↓

> ⚠️ 필기 속 Top-K 설명 중 "확률 분포를 활용하는 것이 아닌 … 빈도수가 가장 높은 단어를 뽑아" 부분은 바로잡자. Top-K 도 **확률 분포를 그대로 활용**한다. 다만 상위 k개로 후보를 자른 뒤 그 안에서 **확률에 비례해 무작위로 뽑는다**는 점이 Greedy 와 다르다.

### 4-2. 💡 보충 : 디코딩 하이퍼파라미터 총정리

| 옵션 | 의미 | 효과 / 주의 |
|---|---|---|
| `do_sample` | `False` = 결정적 탐색(Greedy/Beam), `True` = 확률 추첨(Sampling) | False 면 몇 번을 돌려도 같은 결과 |
| `num_beams` | Beam Search 후보 수 | 1이면 Greedy. 클수록 느리고 무난한 문장 |
| `top_k` | 상위 k개 후보 안에서만 추첨 | 작을수록 안전, 클수록 다양 |
| `top_p` (**Nucleus Sampling**) | 확률을 큰 순서로 더해 **누적 p(예: 0.9)** 가 될 때까지의 단어들만 후보로 | 확신이 높은 자리는 후보가 적고, 애매한 자리는 많아져 **후보 수가 상황에 따라 자동 조절**. 실무에서 top_k 보다 자주 씀 |
| `temperature` | 확률 계산 전 점수를 T로 나눔 : `softmax(점수 / T)` | 0.7~1.0 이 보통. 20.0 은 사실상 무작위 |
| `repetition_penalty` | 이미 나온 토큰의 점수를 이 값으로 나눔 (1.0 = 패널티 없음) | 1.2~1.5 정도. 너무 크면 같은 단어를 꼭 써야 할 때도 피함 |
| `no_repeat_ngram_size` | 같은 **n-gram(연속 n토큰 묶음)** 이 **두 번 나오지 못하게** 막음 | 3이면 '그건 내가 결정할' 같은 3토큰 묶음 재등장 금지 |
| `max_length` / `max_new_tokens` | 전체 길이 상한 / 새로 생성할 토큰 수 상한 | 입력 길이가 다양하면 `max_new_tokens` 가 편함 |
| `min_new_tokens` | 최소 이만큼은 생성 (그 전엔 EOS 금지) | 너무 짧게 끝나는 것 방지 |
| `length_penalty` | Beam Search 에서 길이에 주는 가중 (>1 이면 긴 문장 선호) | Beam 은 짧은 문장을 선호하는 경향이 있어 보정용 |
| `early_stopping` | Beam 후보들이 모두 EOS 를 만나면 바로 종료 | Beam 과 함께 사용 |
| `num_return_sequences` | 돌려받을 결과 개수 | Sampling 과 함께 쓰면 서로 다른 결과 여러 개 |
| `eos_token_id` / `pad_token_id` | 끝 토큰 / 빈칸 토큰 번호 지정 | 경고 메시지가 뜰 때 명시 |
| `penalty_alpha` + `top_k` (**Contrastive Search**) | 확률이 높으면서도 **앞 문맥과 너무 비슷하지 않은** 단어를 고름 | 반복을 줄이면서 일관성도 유지하는 최신 기법 |

> ⚠️ 강의 코드의 `no_repeat_ngram_size` 주석("특정 토큰이 3회 이상 연속 등장하면…")은 정확히는 **"같은 3토큰 묶음이 문장 안에 두 번 나오지 못하게 막는다"** 이다.
> ⚠️ `do_sample=False` + `num_beams=3` 은 Greedy 가 아니라 **Beam Search** 다 (강의 주석의 'Greedy Search 적용'은 '결정적 탐색' 정도로 이해).

In [ ]:
# ===== 4-3. Beam Search + Repetition Penalty =====


def generate_text(input_text, model, tokenizer):
    input_data = tokenizer.encode(input_text, return_tensors='pt')
    outputs = model.generate(input_data,
                             # 입력 포함 최대 150 토큰까지 생성
                             max_length=150,
                             # 무작위 추첨 끄기 (결정적 탐색)
                             do_sample=False,
                             # Beam Search : 후보 문장 3개를 유지하며 탐색
                             num_beams=3,
                             # 이미 등장한 토큰의 점수를 1.5로 나눔
                             repetition_penalty=1.5,
                             # 같은 3-gram 이 다시 나오면 그 토큰의 확률을 0으로
                             no_repeat_ngram_size=3,
                             # 결과 시퀀스 1개
                             num_return_sequences=1)
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return output_text

input_text2 = "다음 최종 프로젝트 주제는 뭘로 하는게 좋을까? 내 생각엔"
print(generate_text(input_text2, model, tokenizer))


# → Beam Search 로 그럴듯한 문장을 고르고, 두 가지 패널티로 반복을 막겠다
# → 수업 결과 : 반복은 사라졌지만 '공모전 328개 작품 접수…' 같은 뉴스 기사체로 흘러가 주제가 이탈했다

#### 📌 `num_return_sequences=1` 인데 왜 여러 문장이 나왔을까?

- **시퀀스(sequence) ≠ 문장(sentence)** 이다. 시퀀스는 **생성된 토큰들의 줄 하나 전체**를 말한다.
- 모형은 EOS 를 만나거나 `max_length=150` 에 닿을 때까지 계속 토큰을 이어 붙이는데, 그 사이에 마침표(`.`)도 그냥 토큰 중 하나라서 **한 시퀀스 안에 문장이 여러 개** 생길 수 있다.
- 3번 기본 설정은 `max_length=50` 이라 짧게 끊겼고, 여기서는 **150** 으로 늘려서 문장이 더 많이 들어갔다.
- **결과물 자체를 여러 개** 받으려면 `num_return_sequences=3` 처럼 바꾼다 (Beam 이면 `num_beams` 이상이어야 하고, 서로 다른 결과를 원하면 `do_sample=True` 와 함께 쓴다).
- **왜 뉴스 기사체로 흘렀나?** KoGPT2 는 뉴스·위키 같은 글을 많이 학습했다. Beam Search 는 **전체 확률이 높은 문장**을 고르는데, '공모전' 같은 단어 뒤에 확률이 가장 높은 흐름이 학습 데이터에 흔했던 **기사 문체**였던 것.

In [ ]:
# ===== 4-4. Top-K + Temperature Sampling =====


def generate_text(input_text, model, tokenizer):
    input_data = tokenizer.encode(input_text, return_tensors='pt')
    outputs = model.generate(input_data,
                             max_length=150,
                             # 샘플링 켜기 : 확률에 비례해 무작위로 단어 선택
                             do_sample=True,
                             # 상위 10개 후보 안에서만 선택 (수업 중 30 → 10 으로 변경해 비교)
                             top_k=10,
                             # 매우 큰 값 → 후보들의 확률이 거의 같아짐 (극단 예시)
                             temperature=20.0,
                             no_repeat_ngram_size=3,
                             repetition_penalty=1.5,
                             num_return_sequences=1)
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return output_text

print(generate_text(input_text2, model, tokenizer))


# → 샘플링이라 실행할 때마다 결과가 달라지고, temperature=20 이라 질문 맥락과 동떨어진 문장이 나오겠다
# → 수업 결과 : '드루토마', '아이언베어', '다크 소드' 같은 게임 공략글이 생성 — 질문과 무관한 엉뚱한 글 (T 가 너무 높은 예)
# → top_k 를 10 / 30 / 100 으로, temperature 를 0.7 / 1.0 / 20 으로 바꿔 다양성과 일관성의 변화를 비교하겠다

---
## 📊 도식화 자료 (p71 ~ p80) — 강사님과 함께 살펴본 그림 정리

오전 실습(0~4번)을 한눈에 정리하고(D01~D06), 오후 의료 멀티모달(5~11번)을 미리 보는(D07~D10) 자료.

### D01. NLP 사전 학습 한눈에 (p71)

![NLP 사전학습 한눈에](images/D01_pretraining_overview.png)

- **위쪽 흐름** : 대규모 텍스트(라벨 없음) → **사전 학습**(자기지도학습, 빈칸 맞히기) → **사전 학습 모델**(언어 지식을 가중치에 저장, 예: `klue/bert-base`, `skt/kogpt2-base-v2`) → **파인튜닝**(적은 라벨 데이터로 재조정 = 전이학습) → **내 과제에 적용**(분류·개체명 인식·QA·검색·챗봇·요약·생성)
- 사전 학습은 **"한 번, 크게"**(비용이 커서 공개 모형 다운로드로 대체), 파인튜닝은 **"과제마다, 작게"**(내 데이터로)
- **아래쪽 3계열** : Transformer 의 어느 쪽을 쓰느냐에 따라
  - **BERT (이해형)** : 인코더만, 양방향 / MLM · NSP / 분류 · 개체명 인식 · 추출형 QA · 검색
  - **GPT (생성형)** : 디코더만, 왼쪽→오른쪽 / 다음 단어 예측 / 대화 · 요약 · 글쓰기 · 추론
  - **T5 · BART (변환형)** : 인코더 + 디코더 모두 / T5 는 모든 과제를 Text-to-Text 로 통일, BART 는 망가뜨린 문장을 원래대로 복원하며 학습 / 기계 번역 · 문서 요약 · 생성형 QA

### D02. BERT vs GPT — 같은 Transformer, 다른 읽는 방향 (p72)

![BERT vs GPT](images/D02_bert_vs_gpt.png)

- **"누가 누구를 참고?" 격자** (행 = 지금 단어, 열 = 참고하는 단어)
  - **BERT** : 모든 칸이 채워짐 → 모든 단어가 앞뒤 모든 단어를 참고 (`[MASK]` 를 앞(1)과 뒤(3·4·5)로 채움)
  - **GPT** : 대각선 아래만 채워지고 위쪽은 ✕ → **미래 칸은 가림(마스크)**. 4번째 단어는 1·2·3만 보고 예측
- 이 삼각형 모양 가림판이 0번 보충의 **Masked Self-Attention** 이다

### D03. BERT 입력 만들기 — 세 가지 임베딩의 합 (p73)

![BERT 입력 만들기](images/D03_bert_input_embeddings.png)

- 두 문장 A(`김치찌개 먹었어`)·B(`커피 못 마셨네`)를 `[CLS] … [SEP] … [SEP]` 한 줄로 넣는다 (NSP 실습 문장)
- 토큰 하나마다 **① 토큰 임베딩**(WordPiece 조각의 의미) **+ ② 세그먼트 임베딩**(`E_A` / `E_B`) **+ ③ 위치 임베딩**(`E_0` ~ `E_8`)을 **더해서** 최종 입력 벡터를 만든다
- `##찌개`, `##마셨네` 의 `##` = 앞 조각에 이어지는 서브워드 (1번 보충 참고)
- 하단 요약 : `[CLS]` = 문장 전체 요약(NSP·분류) / WordPiece = OOV 문제 감소 / Segment = 두 문장 구분(NSP 필수) / Position = Attention 은 순서를 모르므로 자리 정보 추가

### D04. MLM — 가리고 맞히며 배운다 (p74)

![MLM](images/D04_mlm.png)

- ① 원본 문장 → ② 약 15% 를 `[MASK]` 로 가림 → ③ 양방향 인코더가 가려지지 않은 모든 단어를 동시에 참고 → ④ 가려진 자리만 예측
- **손실(오차)은 가려진 자리에서만 계산** → 정답이 원래 단어이므로 사람이 라벨을 만들 필요가 없다 (자기지도학습)
- 오른쪽 : 뽑힌 15% 의 **80 / 10 / 10 규칙** (1-1 보충 참고)
- 아래 실습 결과 : 모델은 정답 하나가 아니라 **사전 전체에 대한 확률 분포**를 낸다 (상위 5개 합 약 47%, 나머지는 다른 단어들에 분산)

### D05. NSP — 두 문장이 이어지는가 (p75)

![NSP](images/D05_nsp.png)

- ① 학습 데이터 : IsNext : NotNext = **50 : 50**. 예) "환자가 심한 두통을 호소했다." → "의사는 CT 촬영을 권했다."(IsNext) / "오늘 주식 시장이 크게 올랐다."(NotNext)
- ② 판별 방식 : `[CLS]` 출력 벡터 → 분류층(출력 2개) → softmax → 0 이어짐 / 1 아님
- 실습 결과 그래프 : 김치찌개·커피 **이어짐 65.7%**, 고백·자연어 처리 **아님 99.7%** (1-5 해석 참고)

### D06. GPT 문장 생성과 디코딩 전략 (p76)

![GPT 디코딩 전략](images/D06_gpt_decoding.png)

- **생성 루프** : ① 입력 문장(encode) → ② GPT(앞 단어만 참고) → ③ 다음 토큰 확률표 → ④ **하나 고르기 = 디코딩 전략** → ⑤ 뒤에 붙이기 → 다시 ①로. `max_length` 에 닿거나 끝 토큰이 나올 때까지 반복 (자기회귀)
- ④ 를 조절하는 5가지 : **Greedy**(매번 1등) / **Beam**(후보 3개 유지) / **반복 억제**(penalty 1.5 · no_repeat_ngram 3) / **Top-K**(상위 K개 중 추첨) / **Temperature**(`softmax(점수 / T)`, T<1 뾰족 · T>1 평탄)
- **실습 결과 비교**
  - 기본 설정 : "그건 내가 결정할 문제지." 무한 반복 → **같은 말 되풀이**
  - Beam 3 + 반복 억제 : 문장은 매끄럽지만 뉴스 기사체로 흘러감 → **반복 해결, 주제 이탈**
  - Top-K + T=20 : 다양한 표현이지만 문장 연결이 두서없음 → **T 가 너무 높은 예**

### D07. 의료 VQA 전이학습 — 3단계로 이어지는 학습 (p77)

![의료 VQA 전이학습](images/D07_medical_vqa_transfer.png)

- 앞 단계에서 학습한 **가중치를 출발점으로 이어받아**, 점점 좁고 전문적인 데이터로 학습한다
  - **1단계 BLIP 사전 학습** (Salesforce) : 웹 이미지-텍스트 1,400만 장(LAION 추가 시 1억 2,900만 장) / **ITC · ITM · LM** 세 목표 / 이미지를 이해하고 문장을 만드는 능력
  - **2단계 VQA 미세조정** (`Salesforce/blip-vqa-base`) : VQA v2.0 일반 사진 질의응답 / 영상 + 질문 → 답변 '생성' / 예) "What color is the bus?" → red
  - **3단계 의료 VQA 미세조정** (`ayyuce/blip-vqa-rad`) : VQA-RAD / 2 epoch · 학습률 5e-5 / 예) "Is there a fracture?" → no
- **왜 전이학습인가** : 의료 데이터는 적다(VQA-RAD 영상 315장). 처음부터 학습하면 부족하므로 **일반 지식 위에 의료 표현만 덧입힌다**
- 노트북에서는 3단계 모형을 불러올 때 **전처리기(토크나이저 · 384px 이미지 변환)는 2단계 것을 재사용**

> 💡 BLIP 사전 학습 3가지 목표 : **ITC**(Image-Text Contrastive, CLIP 처럼 짝 맞는 이미지-문장을 가깝게) · **ITM**(Image-Text Matching, 이 이미지와 문장이 짝인지 yes/no 판별) · **LM**(Language Modeling, 이미지를 보고 문장 생성)

### D08. 테스크와 사용 데이터 — 의료 시각 질의응답 (p78)

![VQA-RAD 데이터](images/D08_vqa_rad_task_data.png)

- **테스크** : 영상 + 질문 → BLIP VQA → 답변. 후보 중에서 '고르는' 분류가 아니라 단어를 하나씩 **'생성'** 하므로 자유 답변(Open)도 가능
- **VQA-RAD** : 방사선 영상 **315장**, 질문-답변 **약 2,248개**, Test 고유 영상 **203장**, **CC0** (자유 이용 라이선스), X-ray · CT · MRI, 임상의가 작성한 질문
- **분할** : Train 1,793 (3단계 미세조정 재료) / Test 451 (평가 전용, 노트북 실습 범위)
- **질문 유형** : **Closed**(yes/no, Test 451개 중 251개) / **Open**(자유 답변, 예: "where are the kidney?" → not seen here)
- **평가 설계** : Closed 앞 50개만, 예측과 정답이 **완전히 같으면 정답**. Open 은 표현이 다양해 문자열 일치 채점이 어려워 제외
- **주의** : 같은 영상에 질문이 여러 개 → Train·Test 영상 겹침 확인 필요 / 공개 미세조정 모형은 학습 데이터 명세가 없어 Test 포함 여부를 알 수 없음 → **정확도는 참고용**

### D09. 모형 구조 — BLIP VQA (p79)

![BLIP VQA 구조](images/D09_blip_vqa_architecture.png)

- **① 이미지 인코더 (ViT-B/16)** : 384×384 영상을 16×16 **패치 576개**로 잘라 + `[CLS]` = 577 토큰 → Self-Attention + FFN × 12층 → **영상 임베딩 577 × 768**
- **② 질문 인코더 (BERT-base + Cross-Attention)** : 질문 토큰(`[ENC]` 신호 + 질문) → Self-Attention(질문 단어끼리) → **Cross-Attention(영상 임베딩 참조)** → FFN × 12층 → **멀티모달 임베딩**
- **③ 답변 디코더 (BERT-base Causal + Cross-Attention)** : `[DEC]` 에서 시작해 **Causal Self-Attention**(앞 단어만, GPT 와 같은 방식) + Cross-Attention(질문+영상 참조) → 다음 단어 확률(어휘 30,524개 중 선택) → `[SEP]` 이 나오면 종료
- **사전 학습과의 연결** : ① = 1단계 ViT 그대로 / ② = ITM 에 쓰던 '이미지 기반 텍스트 인코더' / ③ = LM 에 쓰던 '이미지 기반 텍스트 디코더'

> 💡 **Cross-Attention** = 한 종류의 데이터(질문)가 **다른 종류의 데이터(영상)를 참고**하는 어텐션. Day10 Seq2Seq 의 디코더가 인코더 출력을 보던 Attention 과 같은 원리로, 멀티모달 모형의 핵심 연결 고리다.

### D10. 활용 방안과 실제 모형 점검 (p80)

![활용 방안과 모형 점검](images/D10_usage_and_model_check.png)

- **점검 : 3단계 모형(`ayyuce/blip-vqa-rad`)이 정확도 0.00 이 나온 이유**
  - ① 이미지 인코더 ✓ 불러옴 / **② 질문 인코더 ✕ 체크포인트에 없음** (12층 전체 + 임베딩이 무작위 초기화, MISSING) / ③ 답변 디코더 ✓ 불러옴
  - 근거 : LOAD REPORT 에서 `text_encoder.*` 항목이 모두 MISSING, 파일 크기 990MB(2단계 1.54GB 보다 작음), 모델 카드의 학습 데이터 'unknown'
  - **결과 : 질문을 읽지 못함 → 예측이 전부 빈 문자열 → 정확도 0.00**. 같은 50문항에서 2단계 일반 BLIP 은 0.60 → **0.00 은 의료 성능이 아니라 '불러오기' 문제**
  - **대응** : ① 불러온 직후 LOAD REPORT 의 MISSING 항목부터 확인 ② 세 블록이 모두 들어 있는 의료 VQA 체크포인트로 교체 ③ 또는 blip-vqa-base 를 VQA-RAD Train 1,793 으로 직접 미세조정 → Test 451 로 평가
- **활용 방안** : 판독 보조 질의("흉수가 있나요?" → 판독의 2차 확인) / 영상 분류·정리(장기·모달리티·촬영 방향) / 데이터 라벨링 보조(초벌 라벨 → 사람이 검수) / 판독문 검증(누락·불일치 표시) / 의료 교육(의대생·전공의·방사선사 퀴즈)
- **쓰기 전 조건** : 진단을 대신하지 않음(반드시 판독의 검토) / 그럴듯한 오답(환각) 가능, 영어 질문 기준이라 한국어는 추가 학습 필요 / 자체 데이터로 재평가 후 도입(환자 단위로 학습·평가 분리)
---

## 5. 의료 멀티모달 (Multi-Modal) 사전 학습 모형

- **멀티모달** : 텍스트, 이미지, 음성, 정형 데이터 등 서로 다른 형태(**Modality**)의 데이터를 함께 처리하는 모델
- 의료 데이터는 본질적으로 멀티모달
  - 영상 (X-ray / CT / MRI) + 판독문 (텍스트) + 검사 수치 (정형) + 진료 기록 ...
  - 영상의학과 전문의는 영상을 **"보고"** 판독문을 **"쓰는"** 작업 → 이미지 → 텍스트 변환
- 오전 내용(NLP Pre-Training)과의 연결
  - BERT (Transformer Encoder) : 텍스트를 벡터로 변환 (이해)
  - GPT (Transformer Decoder) : 벡터를 바탕으로 다음 단어를 생성 (생성)
  - **ViT (Vision Transformer)** : 이미지를 작은 조각(**Patch**)으로 잘라 단어 토큰처럼 Transformer Encoder 에 입력
  - 이미지 인코더(ViT) + 텍스트 인코더/디코더(BERT/GPT)를 결합 = **Vision-Language Model (VLM)**
- 멀티모달 사전 학습 모형의 구조

| 구조 | 대표 모형 | 방식 | 활용 |
|---|---|---|---|
| **Dual Encoder** | CLIP | 이미지 인코더와 텍스트 인코더가 각각 벡터를 만들고 같은 벡터 공간에서 비교 | 제로샷 분류, 이미지-텍스트 검색 |
| **Encoder-Decoder** | BLIP | 이미지 인코더가 만든 벡터를 텍스트 디코더가 받아 문장 생성 (GPT 방식) | 이미지 캡셔닝, VQA |
| **Vision Encoder + LLM** | LLaVA, MedGemma | 대형 언어 모델에 이미지 인코더를 연결 → 대화형 판독 보조 | 모델이 크고 GPU 필요 → 실습에서는 개념만 |

- 의료 특화 사전 학습 모형
  - **PubMedCLIP** : ROCO 데이터(방사선 영상 + 논문 캡션, 약 8만 쌍)로 CLIP 을 추가 학습 (6번 실습의 의료 CLIP)
  - **BiomedCLIP** : PubMed Central 논문 그림-캡션 1,500만 쌍(PMC-15M)으로 학습
  - **MedGemma** : Google 의 의료 특화 VLM (4B)

### 데이터 : VQA-RAD (도식화 D08 참고)
- 방사선 영상 (X-ray / CT / MRI) + 질문 + 답변으로 구성된 의료 시각 질의응답 데이터
- 영상 315장, 질의응답 약 2,248개 (Train 1,793 / Test 451), 라이선스 **CC0** (자유 이용)
- 질문 유형 : **Closed** (yes/no, 예: "Is there a fracture?") / **Open** (자유 답변, 예: "What organ is shown?")
- Hugging Face Datasets 에서 바로 다운로드 (`flaviagiammarino/vqa-rad`)
- 같은 영상 1장에 질문이 여러 개 → 영상 기준 중복 제거 필요

In [ ]:
# ===== 5-1. Import & Load VQA-RAD =====


import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
# load_dataset : Hugging Face Hub 의 데이터셋을 다운로드해 불러오는 함수
from datasets import load_dataset
from tqdm.auto import tqdm

# '작성자/데이터셋이름' 으로 VQA-RAD 다운로드 → train / test 로 나뉜 DatasetDict
vqa = load_dataset('flaviagiammarino/vqa-rad')
vqa


# → VQA-RAD 를 내려받아 구성을 확인하겠다 — 수업 결과 train 1,793개 / test 451개, 열은 image · question · answer

In [ ]:
# ===== 5-2. Check Test Sample =====


# Test 데이터만 사용 (451개)
test = vqa['test']
# 첫 번째 샘플의 질문 / 정답 / 영상
print(test[0]['question'])
print(test[0]['answer'])
test[0]['image']


# → 수업 결과 : 'is there evidence of an aortic aneurysm?'(대동맥류 소견이 있는가?) → 'yes', 1024×1291 영상

In [ ]:
# ===== 5-3. Remove Duplicate Images =====


# hashlib : 데이터를 고정 길이 지문(해시)으로 바꾸는 표준 라이브러리
import hashlib

# 중복 없는 영상 목록
img_list = []
# 이미 본 영상의 지문을 저장하는 집합 (set 은 '있나?' 확인이 빠름)
seen = set()
for img in tqdm(test['image'], desc='영상 중복 제거'):
    # img.tobytes() : 픽셀값을 바이트로 / md5().hexdigest() : 그 바이트의 16진수 지문
    key = hashlib.md5(img.tobytes()).hexdigest()
    if key not in seen:
        seen.add(key)
        # convert('RGB') : 흑백(1채널) 영상을 3채널로 — CLIP·BLIP 은 RGB 입력을 기대
        img_list.append(img.convert('RGB'))
print(len(img_list))


# → 픽셀이 같은 영상은 같은 지문이 나오므로 지문으로 중복을 걸러, 질문 451개에서 고유 영상 203장만 남기겠다

In [ ]:
# ===== 5-4. Show Images Function =====


# 여러 장의 영상을 한 줄로 출력하는 함수
def show_images(imgs, titles=None):
    if len(imgs) == 0:
        print('출력할 영상이 없습니다.')
        return
    # 1행 N열 서브플롯 / 영상 1장당 가로 4인치
    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    # 영상이 1장이면 axes 가 배열이 아니므로 항상 배열로 통일
    axes = np.atleast_1d(axes)
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i], cmap='gray')
        # 눈금 숨기기
        ax.axis('off')
        if titles is not None:
            ax.set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()

show_images(img_list[:5])


# → 앞으로 분류·검색 결과를 영상과 함께 보기 위한 공용 출력 함수를 만들겠다

## 6. CLIP (Contrastive Language-Image Pre-training)

- OpenAI (2021) / 인터넷에서 수집한 **이미지-텍스트 약 4억 쌍**으로 사전 학습
- **대조 학습 (Contrastive Learning)**
  - 한 배치에 N개의 (이미지, 캡션) 쌍 → **N × N 유사도 행렬**을 계산
  - 짝이 맞는 쌍(**대각선**)의 유사도는 높이고, 짝이 아닌 쌍의 유사도는 낮추도록 학습
- **자기지도학습의 일종** : 사람이 정답 라벨을 달지 않아도 이미지에 붙어 있는 캡션 자체가 학습 신호 → MLM 이 문장 자체에서 정답을 만든 것과 같은 원리
- 구조 : 이미지 인코더 (**ViT-B/32** : 이미지를 32×32 픽셀 패치로 분할) + 텍스트 인코더 (Transformer)
- 결과적으로 **이미지와 문장이 같은 벡터 공간**에 놓임 → 둘 사이 유사도 계산 가능
- **제로샷 분류 (Zero-shot Classification)**
  - 학습 때 본 적 없는 클래스라도 후보 문장만 주면 분류 가능
  - 이미지 벡터와 가장 유사한 후보 문장 벡터를 정답으로 선택 (예: "a chest x-ray" / "a brain MRI" / "an abdominal CT scan")

| 모형 | 학습 데이터 |
|---|---|
| `openai/clip-vit-base-patch32` (일반 CLIP) | 인터넷 일반 이미지 + 설명 |
| `flaviagiammarino/pubmed-clip-vit-base-patch32` (의료 CLIP) | 일반 CLIP + **방사선 영상과 논문 캡션(PubMed)** 으로 추가 학습 |

In [ ]:
# ===== 6-1. Load General & Medical CLIP =====


from transformers import pipeline

# 'zero-shot-image-classification' : 후보 문장(label) 중 이미지와 가장 잘 맞는 것을 고르는 작업
general = pipeline('zero-shot-image-classification', model='openai/clip-vit-base-patch32')
medical = pipeline('zero-shot-image-classification', model='flaviagiammarino/pubmed-clip-vit-base-patch32')


# → 일반 CLIP 과 의료 CLIP 을 같은 방식으로 불러와 성능을 비교할 준비를 하겠다

In [ ]:
# ===== 6-2. Modality Classification =====


# 후보 문장 = 클래스 : 영상 종류(Modality) 4가지
labels = ['a chest x-ray', 'a brain MRI', 'a brain CT scan', 'an abdominal CT scan']

# candidate_labels : 비교할 후보 문장 목록 → 점수(score) 높은 순으로 반환
print(general(img_list[0], candidate_labels=labels))
print(medical(img_list[0], candidate_labels=labels))


# → 수업 결과 : 일반 CLIP 'a chest x-ray' 0.985, 의료 CLIP 0.9999 — 둘 다 맞혔지만 의료 CLIP 이 더 확신하겠다

**Prompt (후보 문장) 설계**
- pipeline 은 기본적으로 `"This is a photo of {}."` 형태의 문장 틀(**hypothesis_template**)에 후보를 넣어 비교
- 문장 틀에 따라 결과가 달라짐 → 의료 영상에 맞는 표현을 쓰면 성능이 개선될 수 있음
- GPT 의 Hyper-Parameter (Beam / Temperature) 처럼, **CLIP 은 Prompt 가 성능을 좌우**

In [ ]:
# ===== 6-3. Compare Zero-shot with Images =====


# 일반 CLIP vs 의료 CLIP 의 1순위 결과를 영상 제목으로 표시
# template : 후보 단어를 끼워 넣을 문장 틀 ({} 자리에 label 이 들어감)
def compare_zero_shot(imgs, labels, template='This is a photo of {}.'):
    titles = []
    for img in imgs:
        # hypothesis_template : 위 문장 틀 적용 / [0] : 1순위 결과만
        g = general(img, candidate_labels=labels, hypothesis_template=template)[0]
        m = medical(img, candidate_labels=labels, hypothesis_template=template)[0]
        titles.append(f"CLIP : {g['label']} ({g['score']:.2f})\nPubMed : {m['label']} ({m['score']:.2f})")
    show_images(imgs, titles)

# 기본 문장 틀
compare_zero_shot(img_list[:5], labels)
# 의료 영상에 맞는 문장 틀로 변경
compare_zero_shot(img_list[:5], labels, template='a radiology image showing {}.')


# → 5장의 영상에서 두 모형의 판단을 나란히 보고, 문장 틀만 바꿔도 결과가 달라지는지 확인하겠다
# → CLIP 은 '문장'과 비교하므로 질문을 어떻게 쓰느냐(프롬프트)가 성능에 영향을 주겠다

## 7. 의료 제로샷의 한계 : 소견 (Finding) 수준 분류

**질문 (수업 필기)** : 소견 수준 분류가 뭐야? X-ray·CT·MRI 는 모양이 달라서 구분이 되는데, 영상 안에 정상이냐 폐렴이냐 심비대냐처럼 **미세한 질병**이 있는지 없는지는 판단하기 어렵다는 거야? 한계가 명확하게 있다는 거지?

**답 : 맞다.** 정리하면

| 수준 | 질문 | 난이도 | 제로샷 CLIP |
|---|---|---|---|
| **모달리티 (Modality)** | 이게 흉부 X선이야, 뇌 MRI 야? | 쉬움 — 전체 모양·밝기·구도가 확연히 다름 | 잘 맞힘 (6번 결과 0.98~0.99) |
| **소견 (Finding)** | 이 흉부 X선에 폐렴이 있어, 정상이야? | 어려움 — **같은 종류의 영상 안에서 일부 영역의 미세한 차이** | 점수가 고르게 낮거나 한쪽으로 쏠림 → 신뢰 어려움 |

- **소견(Finding)** = 영상의학과 의사가 판독문에 적는 '영상에서 발견된 이상 소견'. 예) 폐렴(pneumonia, 폐 감염으로 뿌옇게 보임), 흉수(pleural effusion, 폐 바깥 흉막강에 물이 참), 심비대(cardiomegaly, 심장 그림자가 커짐)
- **왜 어려운가**
  1. CLIP 은 '이미지 전체'와 '문장 전체'를 벡터 하나씩으로 비교한다 → 작은 병변 같은 **국소적 정보가 전체 벡터에 묻힌다**
  2. 학습 데이터(인터넷/논문 캡션)에 '이 영상의 오른쪽 아래 폐에 폐렴' 같은 **정밀한 소견-영상 짝이 적다**
  3. 해상도를 224px 정도로 줄여 입력하므로 **미세한 질감 차이가 사라진다**
- 강의 정리 : 사전 학습 데이터에 소견 설명이 충분하지 않으면 제로샷으로는 한계 → Fine-Tuning 필요
- **결론** : 제로샷은 **분류·정리용(어떤 영상인지)** 으로는 쓸 만하지만 **진단용(어떤 병인지)** 으로는 한계가 명확하다 → 의료 데이터로 **Fine-Tuning** 이 필요하다

In [ ]:
# ===== 7-1. Collect Chest X-rays =====


# 의료 CLIP 이 흉부 X선으로 분류한 영상만 최대 4장 모으기
chest_imgs = []
for img in img_list:
    if medical(img, candidate_labels=labels)[0]['label'] == 'a chest x-ray':
        chest_imgs.append(img)
    # 4장을 찾으면 반복 중단
    if len(chest_imgs) == 4:
        break
len(chest_imgs)


# → 소견 분류 실험용으로 흉부 X선 4장을 골라내겠다

In [ ]:
# ===== 7-2. Finding-level Zero-shot =====


# 소견 수준의 후보 문장
findings = ['a normal chest x-ray',
            'a chest x-ray with pneumonia',
            'a chest x-ray with pleural effusion',
            'a chest x-ray with cardiomegaly']
# template='{}' : 후보 문장을 그대로 사용
compare_zero_shot(chest_imgs, findings, template='{}')


# → 같은 흉부 X선 안에서 소견을 구분하게 해 보고, 제로샷만으로는 진단 수준 분류가 어렵다는 점을 확인하겠다

## 8. CLIP 임베딩 + 코사인 유사도 : 텍스트로 의료 영상 검색

- CLIP 은 이미지와 텍스트를 **같은 임베딩 공간**에 매핑하므로, 사용자가 입력한 **텍스트와 가장 관련성이 높은 의료 영상을 검색**할 수 있다
- 예) 영상의학에서 디스크 사진이 있을 때, 비슷한 소견의 텍스트나 영상을 찾아오는 테스크
- **글자를 보고 데이터에 있는 이미지를 찾아오는 것** — 영상이 미리 분류(라벨링)되어 있던 게 아니다. 라벨 없이 벡터 거리만으로 찾는다

- 이미지와 텍스트가 **같은 벡터 공간(512차원)** 에 놓여 있어, 서로 다른 모달리티끼리 직접 비교 가능
- 활용 : **판독문으로 유사 증례 영상 검색** / **영상으로 유사 판독문 검색** (검색 기반 리포트 생성)

2번 챗봇과 **완전히 같은 구조**다.

| | 2번 BERT 챗봇 | 8번 CLIP 영상 검색 |
|---|---|---|
| 저장해 둔 벡터 | 질문(Q) 문장 임베딩 | 영상 임베딩 |
| 입력 | 사용자 질문 | 검색 문장 |
| 비교 | 코사인 유사도 | 코사인 유사도 |
| 출력 | 가장 비슷한 질문의 답(A) | 가장 비슷한 영상 top-k |

In [ ]:
# ===== 8-1. Load CLIP Model & Processor =====


# CLIPModel : 이미지 인코더 + 텍스트 인코더 / CLIPProcessor : 토크나이저 + 이미지 전처리기
from transformers import CLIPModel, CLIPProcessor

model_name = 'flaviagiammarino/pubmed-clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(model_name)
clip_proc = CLIPProcessor.from_pretrained(model_name)


# → pipeline 대신 모형을 직접 불러와 영상·문장 벡터를 꺼내 쓰겠다

In [ ]:
# ===== 8-2. Embedding Functions =====


# transformers 버전에 따라 반환 형태가 다름
# 4.x : 텐서를 바로 반환 / 5.x : Output 객체 반환 (벡터는 pooler_output 에 있음)
def to_vector(out):
    return out if torch.is_tensor(out) else out.pooler_output

# 영상 1장 → 벡터
def image_embedding(img):
    # 크기 조정·정규화 후 텐서로 변환
    inputs = clip_proc(images=img, return_tensors='pt')
    with torch.no_grad():
        vec = to_vector(clip_model.get_image_features(**inputs))
    # (1, 512) → (512,) 로 줄이고 numpy 배열로 변환
    return vec.squeeze().numpy()

# 문장 1개 → 벡터
def text_embedding(text):
    # padding=True : 길이를 맞추기 위해 빈 토큰을 채움
    inputs = clip_proc(text=[text], return_tensors='pt', padding=True)
    with torch.no_grad():
        vec = to_vector(clip_model.get_text_features(**inputs))
    return vec.squeeze().numpy()

# 두 벡터 모두 512차원인지 확인 → 같은 공간에 있어야 비교 가능
image_embedding(img_list[0]).shape, text_embedding('a chest x-ray').shape


# → 영상과 문장을 각각 같은 512차원 공간의 벡터로 바꾸는 함수를 만들겠다 — 수업 결과 ((512,), (512,))

In [ ]:
# ===== 8-3. Encode All Images =====


# 모든 영상을 벡터로 변환하여 저장 (2번 챗봇의 df1['embedding'] 과 같은 작업)
df = pd.DataFrame({'image': img_list})
df['embedding'] = [image_embedding(x) for x in tqdm(df['image'], desc='영상 인코딩')]

# 코사인 유사도 함수 (2번과 동일)
def cos_sim(A, B):
    return np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B))

df.shape


# → 검색 대상 영상 203장의 벡터를 미리 계산해 표에 저장하겠다 (수업 때 약 2분, 결과 (203, 2))

In [ ]:
# ===== 8-4. Text-to-Image Search =====


# 검색 문장 Q 와 가장 비슷한 영상 top_k 장을 출력·반환
def return_image(Q, top_k=3):
    # 검색 문장 → 벡터
    embedd = text_embedding(Q)
    # 모든 영상 벡터와 유사도 계산
    df['Score'] = df['embedding'].apply(lambda x: cos_sim(x, embedd))
    # nlargest(k, 열) : 해당 열 값이 큰 순서로 k개 행
    top = df.nlargest(top_k, 'Score')
    show_images(list(top['image']), [f'{Q}\nscore : {s:.3f}' for s in top['Score']])
    return top

result1 = return_image('a chest x-ray')
result2 = return_image('an axial CT scan of the brain')
result3 = return_image('an MRI of the brain')


# → 텍스트를 입력했을 때 해당되는 영상을 찾아오게 하겠다
# → 수업 중 NameError(return_image is not defined)는 함수 정의 셀을 실행하지 않아서였으므로, 함수와 호출을 한 셀에 두었다

**CLIP 유사도 행렬 확인**
- 영상 N장 × 문장 N개의 유사도 행렬 → 대각선(짝이 맞는 쌍)의 값이 높게 나오는지 확인
- **CLIP 이 사전 학습에서 만들려고 했던 결과가 바로 이 행렬**이다 (6번 대조 학습의 N × N 행렬)

In [ ]:
# ===== 8-5. Similarity Matrix =====


# 각 검색 결과의 1순위 영상 3장 × 검색 문장 3개
queries = ['a chest x-ray', 'an axial CT scan of the brain', 'an MRI of the brain']
imgs = [result1['image'].iloc[0], result2['image'].iloc[0], result3['image'].iloc[0]]

# 문장과 영상을 한 번에 전처리
inputs = clip_proc(text=queries, images=imgs, return_tensors='pt', padding=True)
with torch.no_grad():
    # logits_per_image : (영상 수 × 문장 수) 유사도 행렬
    # softmax(dim=-1) : 영상마다(행마다) 3개 문장에 대한 확률로 변환
    probs = clip_model(**inputs).logits_per_image.softmax(dim=-1).numpy()

plt.imshow(probs, cmap='Blues')
plt.xticks(range(3), queries, rotation=20)
plt.yticks(range(3), ['image 1', 'image 2', 'image 3'])
for i in range(3):
    for j in range(3):
        plt.text(j, i, f'{probs[i, j]:.2f}', ha='center', va='center')
plt.colorbar()
plt.show()


# → 대각선(영상 i ↔ 문장 i)이 진하면 영상과 문장의 짝이 잘 맞는다고 해석하겠다
# → 영상 '종류'는 잘 찾지만, 파인튜닝 없이는 척추 협착증·폐렴·뇌의 미세 병변 유무처럼 세부 소견으로는 찾기 힘들다

#### 📌 그럼 Fine-Tuning 은 무엇을 해야 하나? (필기 "뭐해야한데" 보충)

1. **데이터 준비** : 우리 병원 영상 + 그 영상의 **판독문/소견 라벨** 짝을 모은다 (예: 흉부 X선 + "우하엽 폐렴")
2. **학습 목표 정하기** : 검색이면 CLIP 처럼 '짝 맞는 영상-문장은 가깝게' 대조 학습, 분류면 출력층을 붙여 소견 라벨 예측
3. **사전 학습 가중치에서 출발**해 적은 학습률로 몇 epoch 만 추가 학습 (처음부터 학습 X)
4. **환자 단위로 Train / Test 분리** 후 평가 — 같은 환자 영상이 양쪽에 섞이면 점수가 부풀려진다
5. 결과를 판독의가 검토하고 **보조 도구로만** 사용 (도식화 D10 '쓰기 전 조건')

## 9. BLIP (Bootstrapping Language-Image Pre-training) : 이미지 캡셔닝

- Salesforce (2022) / 이미지를 **"이해"** 하는 것뿐 아니라 이미지를 보고 문장을 **"생성"** 할 수 있는 모형
- CLIP 은 이미지–문장의 **유사도만** 계산 (고르기) / BLIP 은 이미지를 보고 **문장을 생성**할 수 있음 (쓰기)
- 구조 : 이미지 인코더 (ViT) + 텍스트 인코더 (BERT) + 텍스트 디코더 (GPT 방식)
- **사전 학습 목표 3가지** (오전 내용과 비교)

| 목표 | 내용 | 오전 내용과 비교 |
|---|---|---|
| **ITC** (Image-Text Contrastive) | 이미지-텍스트 대조 학습 | CLIP 과 동일 |
| **ITM** (Image-Text Matching) | 이미지와 문장이 짝이 맞는지 판별 | **NSP 와 유사한 이진 분류** |
| **LM** (Language Modeling) | 이미지를 보고 캡션을 다음 단어 예측으로 생성 | **GPT 와 동일한 방식** |

- **Bootstrapping (CapFilt)** — 이름의 'B'
  - 웹에서 수집한 캡션은 노이즈가 많음 (이미지와 상관없는 설명, 광고 문구 등)
  - 모델이 직접 새 캡션을 생성(**Captioner**)하고, 부적합한 캡션을 걸러내어(**Filter**) 데이터를 정제한 뒤 다시 학습
- 활용 : 이미지 캡셔닝 / 시각 질의응답 (VQA) / 이미지-텍스트 검색

### 이미지 캡셔닝 (Image Captioning)
- 영상을 입력하면 영상을 설명하는 문장을 생성
- 생성 과정은 GPT 와 동일 → 이전 단어들을 바탕으로 다음 단어를 예측 (Autoregressive)
- **Unconditional** : 영상만 입력 / **Conditional** : 문장의 앞부분(prompt)을 함께 입력하면 뒷부분을 이어서 생성
- 생성 모형이므로 **4번 GPT 디코딩 하이퍼파라미터**(Beam Search, Penalty)를 그대로 쓸 수 있음

> 💡 **필기 보충 : 구조** — 필기의 "이미지 인코더 + 텍스트 인코더" 는 정확히는 캡셔닝 시 **이미지 인코더(ViT) + 이미지를 참고하는 텍스트 '디코더'** 다. 디코더가 Cross-Attention 으로 영상을 보며 GPT 처럼 한 단어씩 문장을 생성한다 (도식화 D09 의 ③번과 같은 구조). 텍스트 '인코더'는 사전 학습의 ITC · ITM 과 VQA 의 질문 이해에 쓰인다.

In [ ]:
# ===== 9-1. Load BLIP Captioning =====


# BlipProcessor : 이미지 전처리 + 토크나이저 / BlipForConditionalGeneration : 캡션 생성 모형
from transformers import BlipProcessor, BlipForConditionalGeneration

cap_name = 'Salesforce/blip-image-captioning-base'
cap_proc = BlipProcessor.from_pretrained(cap_name)
cap_model = BlipForConditionalGeneration.from_pretrained(cap_name)


# → 일반 이미지로 학습된 BLIP 캡셔닝 모형을 불러오겠다

In [ ]:
# ===== 9-2. Caption Function =====


# prompt 가 없으면 Unconditional, 있으면 Conditional 캡셔닝
# **kwargs : num_beams 같은 추가 옵션을 generate 에 그대로 넘기기 위한 문법
def generate_caption(img, prompt=None, **kwargs):
    if prompt is None:
        inputs = cap_proc(images=img, return_tensors='pt')
    else:
        inputs = cap_proc(images=img, text=prompt, return_tensors='pt')
    with torch.no_grad():
        # max_new_tokens=30 : 새로 생성할 토큰 최대 30개 (입력 길이 제외)
        outputs = cap_model.generate(**inputs, max_new_tokens=30, **kwargs)
    return cap_proc.decode(outputs[0], skip_special_tokens=True)


# → 하나의 함수로 두 방식의 캡셔닝과 디코딩 옵션까지 모두 처리하겠다

In [ ]:
# ===== 9-3. Captioning Test =====


# Unconditional 캡셔닝
for img in img_list[:3]:
    print(generate_caption(img))
show_images(img_list[:3])

# Conditional 캡셔닝 : 문장 앞부분을 주고 이어서 생성
for img in img_list[:3]:
    print(generate_caption(img, prompt='a radiology image of'))

# 4번 GPT 하이퍼파라미터 적용 : Beam Search + Repetition Penalty
for img in img_list[:3]:
    print(generate_caption(img, prompt='a radiology image of',
                           num_beams=3, repetition_penalty=1.5, no_repeat_ngram_size=3))


# → 일반 모형이 의료 영상을 얼마나 '의료답게' 설명하는지, 프롬프트와 디코딩 옵션으로 얼마나 나아지는지 비교하겠다
# → 수업 결과 Unconditional : 'a chest xray with a small, white, chest' / 'x - ray - x - ray ...' 반복
# → 수업 결과 Conditional · Beam : 'a radiology image of the chest' — 반복은 줄었지만 소견은 없이 '흉부 사진' 수준의 묘사에 그치겠다

#### 📌 캡셔닝 결과 해석 (강의 자료)
- 일반 캡셔닝 모형은 "흑백 사진", "사람의 가슴" 수준의 묘사는 가능하지만, **의학적 소견은 생성하지 못함**
- 인터넷 이미지 위주로 사전 학습되어 의료 도메인의 지식이 부족
- 의료 리포트 생성은 의료 데이터로 Fine-Tuning 된 모형이 필요하며, 생성 모형의 **환각 (Hallucination)** 에 주의
  - **환각** : 영상에 없는 병변을 그럴듯한 문장으로 만들어 내는 현상 → 임상에서는 반드시 판독의가 검토
- 세 번째 영상의 `x - ray - x - ray ...` 는 3번 GPT 의 반복 퇴화와 같은 현상이고, `repetition_penalty` · `no_repeat_ngram_size` 로 줄어든 것도 같은 원리

## 10. 시각 질의응답 (VQA : Visual Question Answering)

- 실제 이미지가 주어졌을 때 **질문을 하고 답을 하는** 모형. 이미지와 텍스트를 함께 학습하여 이미지에 대한 질문에 답변을 생성한다
- 입력 : **영상 + 질문** → 출력 : **답변** (구조는 도식화 D09)
- BLIP VQA 구조
  1. **이미지 인코더** : 영상을 벡터로 변환
  2. **질문 인코더** : 질문 문장을 인코딩하면서, **Cross-Attention** 으로 영상 정보를 함께 반영
  3. **답변 디코더** : 영상 + 질문 정보를 바탕으로 답변을 생성 (GPT 방식)
- 의료 VQA 예시 : "Is there a fracture?" → yes / no, "What organ is shown?" → lung / brain / liver ...
- `Salesforce/blip-vqa-base` : 일반 이미지 VQA 데이터(VQA v2.0)로 학습된 BLIP (도식화 D07 의 2단계)
- 평가는 정답이 명확한 **Closed 질문(yes/no)** 의 정확도로 계산 — 답변이 yes / no 인 질문만 추출하여, 예측이 정답과 일치하는 비율 (도식화 D08 평가 설계)

In [ ]:
# ===== 10-1. Load BLIP VQA =====


# BlipForQuestionAnswering : 영상 + 질문을 받아 답변을 생성하는 BLIP
from transformers import BlipForQuestionAnswering

vqa_name = 'Salesforce/blip-vqa-base'
vqa_proc = BlipProcessor.from_pretrained(vqa_name)
vqa_model = BlipForQuestionAnswering.from_pretrained(vqa_name)

# 영상과 질문 → 답변 / model, processor 를 인자로 받아 다른 모형에도 재사용
def vqa_answer(img, question, model, processor):
    inputs = processor(images=img.convert('RGB'), text=question, return_tensors='pt')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(outputs[0], skip_special_tokens=True)


# → 모형을 인자로 받도록 만들어, 11번에서 Fine-Tuning 모형을 같은 함수로 비교하겠다

In [ ]:
# ===== 10-2. VQA Test =====


# Test 데이터 5개로 질문 / 정답 / 예측 비교
for i in range(5):
    row = test[i]
    pred = vqa_answer(row['image'], row['question'], vqa_model, vqa_proc)
    print(f"질문 : {row['question']}")
    print(f"정답 : {row['answer']} / 예측 : {pred}")
    print('-' * 50)

# 영상을 보면서 직접 질문해 보기
img = test[0]['image']
print(vqa_answer(img, 'Is this a chest x-ray?', vqa_model, vqa_proc))
print(vqa_answer(img, 'What organ is shown?', vqa_model, vqa_proc))
img


# → 실제 의사 질문에 대한 정답과 모형 예측을 나란히 보고, 내가 만든 질문에도 답하게 해 보겠다
# → 수업 결과 5문항 중 2문항 정답 : 대동맥류 yes ✓, 좌측 경화 no ✗(정답 yes), 폐 실질 이상 yes ✗(정답 no), 가려진 심장 경계 right ✓, 신장 위치 'in lower right' ✗(정답 not seen here)
# → 직접 질문 : 'Is this a chest x-ray?' → yes 는 맞혔지만 'What organ is shown?' → stomach 로 엉뚱하게 답하겠다 (흉부 영상인데 위)

In [ ]:
# ===== 10-3. Closed Question Accuracy =====


# 답변이 yes / no 인 질문만 필터링
# input_columns='answer' : answer 열만 읽어 빠르게 처리 (영상까지 읽지 않음)
closed = test.filter(lambda a: a.lower() in ['yes', 'no'], input_columns='answer')
print(len(closed))

# 앞에서 n개 Closed 질문의 정확도 계산
def closed_accuracy(model, processor, n=50):
    correct = 0
    for i in tqdm(range(n), desc='VQA 평가'):
        row = closed[i]
        pred = vqa_answer(row['image'], row['question'], model, processor)
        # 공백 제거·소문자로 맞춘 뒤 정답과 같으면 1, 다르면 0 을 더함
        correct += int(pred.strip().lower() == row['answer'].lower())
    return correct / n

acc_base = closed_accuracy(vqa_model, vqa_proc)
print('일반 BLIP VQA 정확도 :', acc_base)


# → Closed 질문 251개 중 앞 50개로 일반 BLIP 의 기준 정확도(baseline)를 구하겠다 — 수업 결과 0.6
# → yes/no 는 찍어도 50% 이므로, 0.5 근처라면 사실상 이해를 못 하고 있다고 해석하겠다

## 11. 의료 VQA 데이터로 Fine-Tuning 된 모형과 비교

- `ayyuce/blip-vqa-rad` : `Salesforce/blip-vqa-base` 를 VQA-RAD 로 Fine-Tuning 한 **커뮤니티 공개 모형** (도식화 D07 의 3단계)
- 2번 BERT 챗봇처럼, 사전 학습 모형 + 도메인 데이터 Fine-Tuning 의 효과를 확인하려는 실습
- 주의 (강의 자료)
  - 개인이 공개한 모형으로 모델 카드에 학습 데이터 / 평가 정보가 충분히 기재되어 있지 않음
  - Test 데이터가 학습에 포함되었을 가능성도 배제할 수 없으므로 정확도는 참고용으로만 해석
- 전처리기(토크나이저 · 이미지 전처리)는 원본 모형(`Salesforce/blip-vqa-base`)과 동일한 것을 사용 → `vqa_proc` 재사용
- 수업 : 코드 리뷰는 간단히 하고, 시각 질의응답(10번) 코드와 **비교해 보면서 실행**

> ⚠️ **실제 결과 : 정확도 0.00** (도식화 D10)
> - 원인은 의료 성능이 아니라 **불러오기 문제** — 체크포인트에 **질문 인코더(`text_encoder.*`) 가중치가 없어 무작위로 채워짐(MISSING)** → 질문을 읽지 못해 예측이 전부 빈 문자열
> - 교훈 : 모형을 불러온 직후 **LOAD REPORT 의 MISSING 항목부터 확인**한다. 공개 모형이라고 다 제대로 된 것은 아니다
> - 대응 : 세 블록이 모두 있는 체크포인트로 교체하거나, `blip-vqa-base` 를 VQA-RAD Train(1,793)으로 직접 미세조정 → Test(451)로 평가

In [ ]:
# ===== 11-1. Fine-tuned vs Base =====


# Fine-Tuning 된 모형 다운로드 (전처리기는 원본 모형의 것을 재사용)
ft_model = BlipForQuestionAnswering.from_pretrained('ayyuce/blip-vqa-rad')

# 동일한 질문으로 두 모형 비교
for i in range(5):
    row = closed[i]
    base = vqa_answer(row['image'], row['question'], vqa_model, vqa_proc)
    ft = vqa_answer(row['image'], row['question'], ft_model, vqa_proc)
    print(f"질문 : {row['question']}")
    print(f"정답 : {row['answer']} / 일반 : {base} / Fine-Tuning : {ft}")
    print('-' * 50)


# → 불러올 때 LOAD REPORT 에 text_encoder.* 가 MISSING 으로 나오는지 먼저 확인하겠다
# → 같은 질문에 일반 모형은 yes/no 로 답하고, Fine-Tuning 모형은 빈 답을 내는지 비교하겠다
# → 수업 결과 : LOAD REPORT 에 text_encoder.encoder.layer.{0...11}.* 가 MISSING, 5문항 모두 Fine-Tuning 답이 빈칸

In [ ]:
# ===== 11-2. Accuracy Comparison =====


acc_ft = closed_accuracy(ft_model, vqa_proc)
print('일반 BLIP VQA 정확도 :', acc_base)
print('Fine-Tuning BLIP VQA 정확도 :', acc_ft)


# → 수업 결과 : 일반 0.6 / Fine-Tuning 0.0
# → 숫자만 보고 'Fine-Tuning 하면 나빠진다'고 결론 내지 말고, '왜 이런 값이 나왔는지'를 모형 상태(LOAD REPORT)로 점검하겠다

## 📝 오늘의 정리

| 모형 | 구조 | 하는 일 | 오늘 실습 |
|---|---|---|---|
| **BERT** | 인코더 | 문장 이해 (MLM · NSP) | 빈칸 채우기, 문장 연결 판단 |
| **Sentence-BERT** | 인코더 + 평균 | 문장 → 벡터 | 검색형 챗봇 |
| **GPT** | 디코더 | 문장 생성 | 이어쓰기, 디코딩 전략 비교 |
| **CLIP** | 이미지 · 텍스트 인코더 | 이미지–문장 유사도 | 제로샷 분류, 텍스트→영상 검색 |
| **BLIP** | 이미지 인코더 + 텍스트 인코더/디코더 | 이미지 → 문장 생성, VQA | 캡셔닝, 의료 VQA |

### 🩻 의료 멀티모달 사전 학습 모형 정리 (강의 자료)
- **CLIP (Dual Encoder)** : 이미지 인코더 + 텍스트 인코더 / 대조 학습 → 제로샷 분류, 텍스트 ↔ 영상 검색(코사인 유사도) / 의료 특화 : PubMedCLIP, BiomedCLIP
- **BLIP (Encoder-Decoder)** : ITC(대조 학습) + ITM(짝 맞추기, NSP 와 유사) + LM(문장 생성, GPT 와 유사) → 이미지 캡셔닝, VQA
- **Vision Encoder + LLM** : LLaVA-Med, CheXagent, MedGemma (`google/medgemma-1.5-4b-it`, 사용 동의 필요 / GPU 권장) → 대화형 판독 보조, 리포트 초안 생성
- **한계**
  - 일반 모형은 의료 소견을 모름 → 도메인 사전 학습 / Fine-Tuning 이 필요
  - 생성 모형의 환각(Hallucination) → 의료 현장에서는 반드시 전문의 검토
  - 대부분 영어로 학습 → 한국어 질문/판독문에는 추가 학습이 필요

### 💡 핵심 포인트
1. 사전 학습 모형은 **Hugging Face 이름 한 줄**로 불러와 바로 쓸 수 있다. 단, **LOAD REPORT 의 MISSING** 은 반드시 확인한다.
2. BERT 입력 = **토큰 + 세그먼트 + 위치 임베딩의 합**, 서브워드(WordPiece)로 OOV 문제를 줄인다.
3. **임베딩 + 코사인 유사도** 패턴은 텍스트 챗봇(2번)과 의료 영상 검색(8번)에 똑같이 쓰인다.
4. 생성 품질은 **디코딩 하이퍼파라미터**에 크게 좌우된다 — 반복(Greedy) ↔ 주제 이탈(Beam) ↔ 두서없음(높은 Temperature) 사이의 균형.
5. 일반 모형의 제로샷은 **모달리티 구분은 되지만 소견 수준 진단은 어렵다** → 도메인 데이터 **전이학습(Fine-Tuning)** 이 필요하고, 의료 AI 는 **보조 도구**로만 쓴다.